# Gradient-isolated M2 + K=1 M4 결합 — Dunnhumby seed 43

seed 42에서 양의 방향을 보인 gradient-isolated M2를 수정하지 않고, K=1 개인별 경제구간 적합도 양성행 가중 M4와 결합합니다.

- 재사용: 동일 개발분할·K=1·seed 43의 M1, 실제 M4
- 신규 학습: M2-GI, M2-GI+M4
- 학습: DAY 1~683 / 개발평가: DAY 684~690
- 신규상품 추천, binary graph, MIN_ITEM_INTER=1, 균등 음성 1개, 100 epoch
- test·holdout·외부 재정렬은 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '3948b3394a5780d10b8ae48289e1373c0bcffd7c'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
!if [ ! -d "$REPO_DIR/.git" ]; then git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git "$REPO_DIR"; fi
!git -C "$REPO_DIR" fetch -q origin $REVIEWED_SHA
!git -C "$REPO_DIR" checkout -q $REVIEWED_SHA
%cd /content/clv-m2-lightgcn-runner
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA

In [ ]:
import json
import torch
import lightgcn_clv_gradient_isolated_m4_k1_combo_screen as combo

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert combo.CODE_VERSION == 'm5-gradient-isolated-m2-personalized-m4-k1-seed43-screen-v1'
cfg = combo.configure_combo_screen()
summary = combo.preflight_summary(cfg)
assert cfg.seed == 43 and cfg.negative_count == 1
assert summary['trained_models'] == [combo.M2_MODEL_ID, combo.M5_MODEL_ID]
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = combo.run_combo_screen(cfg)

In [ ]:
import json
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = [
    'model_id', 'training_origin', 'recall@10', 'ndcg@10',
    'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10', 'vndcg@10',
    'coverage@10', 'user_value_tendency_recommended_price_alignment',
]
print('1) M1·M2-GI·M4·M5-GI 핵심 절대지표')
show(result_df[[column for column in core if column in result_df.columns]])
print('2) 모든 절대지표')
show(result_df)
print('3) 모든 비교')
show(result_df.attrs['comparison'])
print('4) 사전 고정 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))